In [7]:
from transformers import pipeline, AutoTokenizer, AutoModelForSequenceClassification
import pandas as pd
import torch
import torch.nn.functional as f

model_name = "joeddav/xlm-roberta-large-xnli"

tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=False, local_files_only=False)
model = AutoModelForSequenceClassification.from_pretrained(model_name)

classifier = pipeline(
    "zero-shot-classification",
    model=model,
    tokenizer=tokenizer
)

Some weights of the model checkpoint at joeddav/xlm-roberta-large-xnli were not used when initializing XLMRobertaForSequenceClassification: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing XLMRobertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing XLMRobertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Device set to use cuda:0


In [27]:
import re
from collections import defaultdict
from converter import OUTPUT_FILES as FILES

labels = [
    "Нейтральная тональность",
    "Позитивная тональность",
    "Негативная тональность"
]

def clean_text(text):
    if not isinstance(text, str):
        return text

    text = re.sub(r"https?://\S+|www\.\S+", "", text)
    text = re.sub(r"\+?\d[\d\-\s\(\)]{7,}\d", "", text)
    text = re.sub(r"#\w+", "", text)
    text = re.sub(r"@\w+", "", text)
    text = re.sub(r"\s+", " ", text).strip()

    return text

def classify_text_auto(text):
    # sentences = [p for p in text.split(". ") if p.strip()]
    #
    # if not sentences:
    #     sentences = [text]
    #
    # score_sum = defaultdict(float)
    #
    # for s in sentences:
    #     s = clean_text(s)
    #
    #     if len(s) > 8:
    #         res = classifier(s, candidate_labels=labels)
    #
    #         for label, score in zip(res["labels"], res["scores"]):
    #             score_sum[label] += score
    #
    # best_label = max(score_sum, key=score_sum.get)
    #
    # return best_label

    res = classifier(clean_text(text), labels)
    return res["labels"][0]

for csv in FILES:
    df = pd.read_csv(csv, sep=",", quotechar='"')
    df["ТОНАЛЬНОСТЬ"] = df["ТЕКСТ ПОСТА"].apply(classify_text_auto)
    new_name = f"{csv.replace('csv', 'out1').split('.')[0]}_С_Тональностью.csv"
    df.to_csv(new_name, index=False, encoding="utf-8-sig")
    print(f"Сохранен файл {new_name}")

Сохранен файл data/out1/Москва_Мое_сокровище_С_Тональностью.csv
Сохранен файл data/out1/Московская_область_С_Тональностью.csv
Сохранен файл data/out1/Подарок_малышу_Челябинск_С_Тональностью.csv
Сохранен файл data/out1/Татарстан_С_Тональностью.csv
Сохранен файл data/out1/Ульяновская_область_С_Тональностью.csv
Сохранен файл data/out1/Шкатулка_Расту_в_Югре_С_Тональностью.csv
Сохранен файл data/out1/Ямал_С_Тональностью.csv


In [3]:
model.eval()

labels = {
    "positive": "Это положительный отзыв.",
    "neutral":  "Это нейтральное сообщение.",
    "negative": "Это отрицательный отзыв."
}

def predict_sentiment(text):
    scores = {}
    for label, hypo in labels.items():
        inputs = tokenizer(text, hypo, return_tensors="pt", truncation=True)
        with torch.no_grad():
            logits = model(**inputs).logits
            prob = f.softmax(logits, dim=1)[0][0].item()
        scores[label] = prob

    return max(scores, key=scores.get)